# 1. Check GPU

In [ ]:
#@title 1. Check GPU
import subprocess, sys
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode != 0:
    sys.exit(
        "NO GPU ATTACHED.\n"
        "Runtime > Change runtime type > Hardware accelerator > T4 GPU, then rerun.\n"
        "Without this the model loads onto CPU and each request takes minutes."
    )
print(out.stdout)


# 2. Mount Drive and cache weights there

In [ ]:
#@title 2. Mount Drive and cache weights there
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# A fresh Colab session otherwise re-downloads ~7GB of weights, which is
# several minutes of dead air before the demo can start.
HF_HOME = "/content/drive/MyDrive/hf-cache"
os.environ["HF_HOME"] = HF_HOME
Path(HF_HOME).mkdir(parents=True, exist_ok=True)

cached = list(Path(HF_HOME).glob("hub/models--Qwen*"))
print(f"HF_HOME = {HF_HOME}")
print("CACHE HIT - weights already on Drive" if cached else "CACHE MISS - first run will download ~7GB")


# 3. Clone the repo and install

In [ ]:
#@title 3. Clone the repo and install
REPO_URL = "https://github.com/ziad7amoda/target-ocr-mvp.git"  #@param {type:"string"}
BRANCH = "master"  #@param {type:"string"}

import os, shutil
if os.path.exists("/content/app-repo"):
    shutil.rmtree("/content/app-repo")
!git clone --branch {BRANCH} --depth 1 {REPO_URL} /content/app-repo
%cd /content/app-repo
!pip install -q -r requirements.txt -r requirements-gpu.txt
print("installed")


# 4. Start the server and wait for the model

`MODEL_ID` and `LOAD_IN_8BIT` below default to `MBZUAI/AIN` (Arabic-specialised, 7B, loaded in 8-bit to fit a 16GB T4). If AIN underperforms or is too slow during a live demo, the fast fallback is switching these two `#@param` fields — `MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"` and `LOAD_IN_8BIT = False` — and rerunning this cell, not editing code.

**Timing:** a cold cache (no prior Drive cache from cell 2) downloads roughly 15GB of AIN weights and then quantises them to 8-bit on load — realistically 15-25 minutes end to end. A warm Drive cache from an earlier session cuts this to a couple of minutes. `READY_TIMEOUT_MIN` below defaults to 30; raise it if you are on a slow connection. The cell prints a progress line (elapsed time, cache size on disk) while it waits, and fails immediately — not after the timeout — if the server thread actually crashes, rather than just loading slowly.

In [ ]:
#@title 4. Start the server and wait for the model
MODEL_ID = "MBZUAI/AIN"  #@param {type:"string"}
LOAD_IN_8BIT = True  #@param {type:"boolean"}
READY_TIMEOUT_MIN = 30  #@param {type:"integer"}
# A cold cache means downloading ~15GB of AIN weights, then quantising them
# to 8-bit on load - realistically 15-25 minutes. A warm Drive cache (cell
# 2) cuts this to a couple of minutes. Raise this if your connection is slow.

import os

# Must be set BEFORE app.main (and therefore app.config) is imported, since
# Settings() reads the environment at import time.
os.environ["MODEL_ID"] = MODEL_ID
os.environ["LOAD_IN_8BIT"] = str(LOAD_IN_8BIT)

import logging
import threading
import time
import traceback
from pathlib import Path

import requests
import uvicorn

from app.main import app

# --- capture a crash instead of just timing out -------------------------
# The model loads inside FastAPI's lifespan startup, which runs inside
# uvicorn's event loop in this background thread. uvicorn swallows a
# failed lifespan startup itself (it logs the exception and exits the
# lifespan cleanly instead of letting it escape uvicorn.run()), so a bare
# try/except around uvicorn.run() would NOT see a bad model id, an OOM, or
# a bitsandbytes/dtype error - it would look identical to "still loading"
# and burn the whole timeout. Capture the traceback uvicorn already logs
# via a handler, and also catch what does propagate out of uvicorn.run()
# itself (e.g. sys.exit() on startup failure, or a port already in use).
_startup_error = {"traceback": None}


class _LifespanErrorCapture(logging.Handler):
    def emit(self, record):
        if record.exc_info:
            _startup_error["traceback"] = "".join(
                traceback.format_exception(*record.exc_info)
            )


logging.getLogger("uvicorn.error").addHandler(_LifespanErrorCapture())

_thread_error = {"traceback": None}


def _serve():
    try:
        uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")
    except BaseException:
        # BaseException, not Exception: a failed lifespan startup makes
        # uvicorn call sys.exit(), which raises SystemExit out of
        # uvicorn.run() itself.
        _thread_error["traceback"] = traceback.format_exc()


server_thread = threading.Thread(target=_serve, daemon=True)
server_thread.start()


def _cache_dir():
    # HF_HOME is set by cell 2 (Drive cache) - do not hardcode a Drive path
    # here so this also works if that cell is skipped or changed.
    home = os.environ.get("HF_HOME")
    return Path(home) if home else None


def _cache_size_bytes():
    """Best-effort size of the HF cache on disk, so the user can see the
    download growing instead of staring at a blank cell."""
    cache_dir = _cache_dir()
    if cache_dir is None or not cache_dir.exists():
        return None
    total = 0
    for p in cache_dir.rglob("*"):
        try:
            if p.is_file():
                total += p.stat().st_size
        except OSError:
            continue
    return total


def _fmt_bytes(n):
    if n is None:
        return "unknown (cache dir not created yet)"
    size = float(n)
    for unit in ("B", "KB", "MB", "GB"):
        if size < 1024:
            return f"{size:.1f}{unit}"
        size /= 1024
    return f"{size:.1f}TB"


def _crash_message():
    tb = _startup_error["traceback"] or _thread_error["traceback"]
    if tb:
        return tb
    return "(server thread exited but no traceback was captured)"


start = time.time()
start_size = _cache_size_bytes()
deadline = start + READY_TIMEOUT_MIN * 60
ready = False

while time.time() < deadline:
    if _startup_error["traceback"] or _thread_error["traceback"] or not server_thread.is_alive():
        print()
        raise RuntimeError(
            "Server crashed while loading the model - see traceback below:\n\n"
            + _crash_message()
        )

    try:
        h = requests.get("http://127.0.0.1:8000/api/health", timeout=5).json()
        if h.get("loaded"):
            print()
            print(h)
            ready = True
            break
    except Exception:
        pass

    elapsed_min = (time.time() - start) / 60
    print(
        f"\rwaiting for model... {elapsed_min:.1f} min elapsed, "
        f"cache on disk: {_fmt_bytes(_cache_size_bytes())}          ",
        end="",
        flush=True,
    )
    time.sleep(5)

if not ready:
    end_size = _cache_size_bytes()
    growing = (
        start_size is not None and end_size is not None and end_size > start_size
    )
    print()
    raise RuntimeError(
        f"Model did not become ready within {READY_TIMEOUT_MIN} minutes.\n"
        f"Cache dir was {'still growing' if growing else 'NOT growing'} "
        f"({_fmt_bytes(start_size)} -> {_fmt_bytes(end_size)}).\n\n"
        "What to do next:\n"
        "  1. Run `python scripts/bringup.py <image>` in a local/Colab "
        "terminal - it loads the model synchronously, outside a background "
        "thread, and will print the real error instead of hanging.\n"
        "  2. For a fast fallback, set MODEL_ID back to "
        "\"Qwen/Qwen2.5-VL-3B-Instruct\" and LOAD_IN_8BIT = False above, "
        "then rerun this cell."
    )


# 5. Open the public HTTPS tunnel

In [ ]:
#@title 5. Open the public HTTPS tunnel
# Quick tunnel rather than ngrok: no account, no auth token, one less thing
# to fail live.
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

import re, subprocess, threading, time

url = None
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

def _watch():
    global url
    for line in proc.stdout:
        m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
        if m and not url:
            url = m.group(0)

threading.Thread(target=_watch, daemon=True).start()
for _ in range(60):
    if url:
        break
    time.sleep(1)

print("\n" * 2 + "=" * 72)
print("   OPEN THIS URL:")
print(f"   {url}")
print("=" * 72 + "\n" * 2)
print("Camera capture needs HTTPS, which this tunnel provides.")


# 6. Smoke test before sharing your screen

This cell needs an image you supply. No sample photo ships in the repo (`eval/samples/*.jpg` is gitignored). Upload a card photo to this Colab session and set `IMAGE` below to its path, or run the sample generator in `eval/make_samples.py` after installing the Noto Naskh Arabic font.

In [ ]:
#@title 6. Smoke test before sharing your screen
# NOTE: this cell's output contains extracted field values (name, ID
# number, dates, ...). That is the point of a smoke test, but it also
# means those values persist inside a saved .ipynb - clear this cell's
# output before saving or committing the notebook.
import json, os, time, requests

IMAGE = "/content/app-repo/eval/samples/synthetic_01.jpg"  #@param {type:"string"}

if not os.path.exists(IMAGE):
    print("*" * 72)
    print("! NO SAMPLE IMAGE FOUND.")
    print(f"! {IMAGE} does not exist - no sample card ships in the repo")
    print("! (eval/samples/*.jpg is gitignored).")
    print("!")
    print("! To run this smoke test, either:")
    print("!   1. Upload a card photo to this Colab session (folder icon")
    print("!      on the left) and set IMAGE above to its path, or")
    print("!   2. Install the Noto Naskh Arabic font and run the sample")
    print("!      generator (eval/make_samples.py) to create one.")
    print("*" * 72)
else:
    t0 = time.time()
    r = requests.post(
        "http://127.0.0.1:8000/api/extract",
        files={"image": open(IMAGE, "rb")},
        # AIN in 8-bit is slower than the ~10s the old default budgeted
        # for - 7B, and int8 decode carries overhead - so give it real
        # headroom rather than a tight timeout.
        timeout=600,
    )
    print(f"HTTP {r.status_code} in {time.time() - t0:.1f}s")
    body = r.json()
    print(json.dumps(body["fields"], indent=2, ensure_ascii=False))
    print(f"agreement {body['agreement']}  elapsed_ms {body['elapsed_ms']}")